In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import re
from tqdm import tqdm
import os

In [2]:
# Cấu hình
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
}

BASE_URL = 'https://www.dienmayxanh.com'
MIN_DELAY = 0.5
MAX_DELAY = 1.5

## 1. Lấy danh sách Categories

In [3]:
def get_dmx_categories():
    """
    Danh sách các category từ Điện Máy Xanh - Vào Bếp
    """
    categories = [
        # Món theo nguyên liệu
        {'name': 'Món Thịt Heo', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-thit-heo'},
        {'name': 'Món Thịt Bò', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-thit-bo'},
        {'name': 'Món Gà Vịt', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-ga-vit'},
        {'name': 'Món Hải Sản', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-hai-san'},
        {'name': 'Món Cá', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-ca'},
        {'name': 'Món Chay', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chay'},
        {'name': 'Món Rau Củ', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-rau-cu'},
        {'name': 'Món Đậu Hũ', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-dau-hu'},
        {'name': 'Món Trứng', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-trung'},
        
        # Món theo loại
        {'name': 'Món Canh', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-canh'},
        {'name': 'Món Kho', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-kho'},
        {'name': 'Món Xào', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-xao'},
        {'name': 'Món Chiên', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-chien'},
        {'name': 'Món Nướng', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nuong'},
        {'name': 'Món Hấp', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-hap'},
        {'name': 'Món Luộc', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-luoc'},
        {'name': 'Món Gỏi', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-goi'},
        {'name': 'Món Cuốn', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-cuon'},
        {'name': 'Món Lẩu', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-lau'},
        
        # Món ăn sáng, ăn vặt
        {'name': 'Món Ăn Sáng', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-an-sang'},
        {'name': 'Món Ăn Vặt', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-an-vat'},
        {'name': 'Món Bánh', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-banh'},
        {'name': 'Món Chè', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-che'},
        {'name': 'Đồ Uống', 'url': 'https://www.dienmayxanh.com/vao-bep/do-uong'},
        
        # Món theo vùng miền
        {'name': 'Món Miền Bắc', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-mien-bac'},
        {'name': 'Món Miền Trung', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-mien-trung'},
        {'name': 'Món Miền Nam', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-mien-nam'},
        {'name': 'Món Hàn Quốc', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-han-quoc'},
        {'name': 'Món Nhật Bản', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-nhat-ban'},
        {'name': 'Món Trung Quốc', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-trung-quoc'},
        {'name': 'Món Thái', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-thai-lan'},
        {'name': 'Món Âu', 'url': 'https://www.dienmayxanh.com/vao-bep/mon-au'},
    ]
    return categories

categories = get_dmx_categories()
print(f"Tổng số category: {len(categories)}")

Tổng số category: 32


## 2. Lấy danh sách URL công thức

In [4]:
def get_recipe_urls_from_category_dmx(category_url, category_name, max_pages=30):
    """
    Lấy danh sách URL công thức từ Điện Máy Xanh
    """
    recipes = []
    seen_urls = set()
    
    for page in range(1, max_pages + 1):
        try:
            # DMX sử dụng format /page-{n}
            if page == 1:
                url = category_url
            else:
                url = f"{category_url}/page-{page}"
            
            response = requests.get(url, headers=HEADERS, timeout=15)
            
            if response.status_code != 200:
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Tìm các recipe links
            recipe_links = soup.select('a.main-contain, li.item a[href*="/vao-bep/"]')
            
            if not recipe_links:
                recipe_links = soup.find_all('a', href=re.compile(r'/vao-bep/[^/]+-\d+'))
            
            if not recipe_links:
                break
            
            page_urls = []
            for link in recipe_links:
                href = link.get('href', '')
                # Chỉ lấy link chi tiết món ăn (có số ID ở cuối)
                if href and re.search(r'/vao-bep/[^/]+-\d+$', href):
                    if not href.startswith('http'):
                        href = BASE_URL + href
                    
                    if href not in seen_urls:
                        seen_urls.add(href)
                        page_urls.append(href)
            
            if not page_urls:
                break
            
            for recipe_url in page_urls:
                recipes.append((category_name, recipe_url))
            
            print(f"    Page {page}: Found {len(page_urls)} recipes")
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
            
        except Exception as e:
            print(f"    [ERROR] Page {page}: {e}")
            break
    
    return recipes

In [5]:
# Test
test_cat = categories[0]
print(f"Testing: {test_cat['name']}")
test_urls = get_recipe_urls_from_category_dmx(test_cat['url'], test_cat['name'], max_pages=2)
print(f"Found {len(test_urls)} recipes")
for cat, url in test_urls[:3]:
    print(f"  - {url}")

Testing: Món Thịt Heo
    Page 1: Found 22 recipes
    Page 1: Found 22 recipes
Found 22 recipes
  - https://www.dienmayxanh.com/vao-bep/cach-lam-goi-du-du-tom-kho-hap-dan-don-gian-14052
  - https://www.dienmayxanh.com/vao-bep/2-cach-nau-lau-bo-nam-thom-ngon-de-lam-tai-nha-cho-cac-bua-10409
  - https://www.dienmayxanh.com/vao-bep/cach-lam-thach-tu-dua-thom-khom-chua-ngot-deo-ngon-thanh-mat-09987
Found 22 recipes
  - https://www.dienmayxanh.com/vao-bep/cach-lam-goi-du-du-tom-kho-hap-dan-don-gian-14052
  - https://www.dienmayxanh.com/vao-bep/2-cach-nau-lau-bo-nam-thom-ngon-de-lam-tai-nha-cho-cac-bua-10409
  - https://www.dienmayxanh.com/vao-bep/cach-lam-thach-tu-dua-thom-khom-chua-ngot-deo-ngon-thanh-mat-09987


In [6]:
# Crawl tất cả categories
all_recipe_urls = []
seen_urls = set()

for cat in tqdm(categories, desc="Crawling categories"):
    print(f"\n📂 {cat['name']}")
    recipes = get_recipe_urls_from_category_dmx(cat['url'], cat['name'], max_pages=20)
    
    for cat_name, url in recipes:
        if url not in seen_urls:
            seen_urls.add(url)
            all_recipe_urls.append((cat_name, url))
    
    print(f"   ✅ Total: {len(all_recipe_urls)}")
    time.sleep(random.uniform(1, 2))

print(f"\n🎉 Total unique recipes: {len(all_recipe_urls)}")

Crawling categories:   0%|          | 0/32 [00:00<?, ?it/s]


📂 Món Thịt Heo
    Page 1: Found 22 recipes
    Page 1: Found 22 recipes
   ✅ Total: 22
   ✅ Total: 22


Crawling categories:   3%|▎         | 1/32 [00:03<01:57,  3.78s/it]


📂 Món Thịt Bò
    Page 1: Found 22 recipes
    Page 1: Found 22 recipes
   ✅ Total: 22
   ✅ Total: 22


Crawling categories:   6%|▋         | 2/32 [00:06<01:42,  3.42s/it]

...


📂 Món Thái
    Page 1: Found 22 recipes
    Page 1: Found 22 recipes
   ✅ Total: 269
   ✅ Total: 269


Crawling categories:  97%|█████████▋| 31/32 [01:36<00:02,  2.90s/it]


📂 Món Âu
    Page 1: Found 22 recipes
    Page 1: Found 22 recipes
   ✅ Total: 269
   ✅ Total: 269


Crawling categories: 100%|██████████| 32/32 [01:40<00:00,  3.15s/it]


🎉 Total unique recipes: 269


In [7]:
# Lưu URLs
urls_df = pd.DataFrame(all_recipe_urls, columns=['category', 'url'])
urls_df.to_csv('dmx_recipe_urls.csv', index=False)
print(f"Saved {len(urls_df)} URLs")

Saved 269 URLs


## 3. Crawl chi tiết công thức

In [8]:
def get_recipe_detail_dmx(url, category):
    """
    Crawl chi tiết công thức từ Điện Máy Xanh (dienmayxanh.com/vao-bep)
    
    Cấu trúc HTML thực tế:
    - Title: h1 trong div.detail-content
    - Description: div.leadpost p
    - Nguyên liệu: div.staple span (span chứa tên + small chứa số lượng)
    - Các bước: div.method ul li (label là số, h3 là tên bước, p là nội dung)
    - Thời gian: ul.ready li span
    - Số người: div.staple h2 small
    """
    result = {
        "link": url,
        "type_of_food": category,
        "title": None,
        "description": None,
        "author_name": None,
        "cook_time": None,
        "num_of_people": None,
        "calories": None,
        "num_of_ingredients": None,
        "ingredients": [],
        "step": [],
        "note": [],
        "post_date": None,
        "source": "dienmayxanh.com"
    }
    
    def safe_text(node):
        return node.get_text(strip=True) if node else None
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Tìm vùng nội dung chính
        detail_content = soup.find('div', class_='detail-content')
        if not detail_content:
            detail_content = soup  # Fallback to toàn bộ page
        
        # Title - h1 đầu tiên trong detail-content
        try:
            title_tag = detail_content.find('h1')
            if title_tag:
                result['title'] = safe_text(title_tag)
        except:
            pass
        
        # Description - div.leadpost p
        try:
            leadpost = detail_content.find('div', class_='leadpost')
            if leadpost:
                result['description'] = safe_text(leadpost)
            else:
                # Fallback to meta description
                meta_desc = soup.find('meta', attrs={'name': 'description'})
                if meta_desc:
                    result['description'] = meta_desc.get('content', '')
        except:
            pass
        
        # Author - thử tìm nhiều vị trí
        try:
            # Từ script JSON-LD
            script_tag = soup.find('script', type='application/ld+json')
            if script_tag:
                import json
                try:
                    data = json.loads(script_tag.string)
                    if isinstance(data, dict) and 'author' in data:
                        author_data = data.get('author', {})
                        if isinstance(author_data, dict):
                            result['author_name'] = author_data.get('name')
                        elif isinstance(author_data, str):
                            result['author_name'] = author_data
                except:
                    pass
            
            if not result['author_name']:
                author_tag = soup.find('span', class_='author') or soup.find('a', class_='author')
                result['author_name'] = safe_text(author_tag)
        except:
            pass
        
        # Thông tin thời gian, số người từ ul.ready
        try:
            ready_items = soup.select('ul.ready li')
            for item in ready_items:
                h2_tag = item.find('h2')
                span_tag = item.find('span')
                if h2_tag and span_tag:
                    label = safe_text(h2_tag).lower()
                    value = safe_text(span_tag)
                    if 'chuẩn bị' in label or 'chế biến' in label:
                        if result['cook_time']:
                            result['cook_time'] += ' + ' + value
                        else:
                            result['cook_time'] = value
        except:
            pass
        
        # Số người - từ div.staple h2 small
        try:
            staple_div = soup.find('div', class_='staple')
            if staple_div:
                h2_tag = staple_div.find('h2')
                if h2_tag:
                    small_tag = h2_tag.find('small')
                    if small_tag:
                        result['num_of_people'] = safe_text(small_tag)
        except:
            pass
        
        # Ingredients - div.staple span
        try:
            ingredients = []
            staple_div = soup.find('div', class_='staple')
            if staple_div:
                ing_spans = staple_div.find_all('span', recursive=False)
                for span in ing_spans:
                    # Lấy tên nguyên liệu
                    name_parts = []
                    for content in span.contents:
                        if isinstance(content, str):
                            text = content.strip()
                            if text:
                                name_parts.append(text)
                    
                    name = ' '.join(name_parts).strip()
                    
                    # Lấy số lượng từ small
                    small_tag = span.find('small')
                    quantity = safe_text(small_tag) if small_tag else ''
                    
                    # Lấy ghi chú từ em
                    em_tag = span.find('em')
                    note = safe_text(em_tag) if em_tag else ''
                    
                    # Ghép lại
                    if name:
                        ingredient_text = name
                        if quantity:
                            ingredient_text += f" - {quantity}"
                        if note:
                            ingredient_text += f" ({note})"
                        ingredients.append(ingredient_text.strip())
            
            # Fallback: thử các selector khác
            if not ingredients:
                for selector in ['div.ingredient li', 'ul.ingredient li', 'div.nguyenlieu li']:
                    items = soup.select(selector)
                    if items:
                        for item in items:
                            text = safe_text(item)
                            if text and len(text) > 1:
                                ingredients.append(text)
                        break
            
            result['ingredients'] = ingredients
            result['num_of_ingredients'] = len(ingredients)
        except:
            pass
        
        # Steps - div.method ul li
        try:
            steps = []
            method_div = soup.find('div', class_='method')
            if method_div:
                step_items = method_div.find_all('li')
                for item in step_items:
                    # Lấy số bước từ label
                    label_tag = item.find('label')
                    step_num = safe_text(label_tag) if label_tag else ''
                    
                    # Lấy tiêu đề bước từ h3
                    h3_tag = item.find('h3')
                    step_title = safe_text(h3_tag) if h3_tag else ''
                    
                    # Lấy nội dung từ div.text-method
                    text_div = item.find('div', class_='text-method')
                    if text_div:
                        # Lấy tất cả p tags
                        p_tags = text_div.find_all('p')
                        contents = []
                        for p in p_tags:
                            text = safe_text(p)
                            if text:
                                contents.append(text)
                        step_content = ' '.join(contents)
                    else:
                        step_content = safe_text(item)
                    
                    # Ghép thành 1 bước
                    if step_content:
                        if step_num and step_title:
                            step_text = f"Bước {step_num} - {step_title}: {step_content}"
                        elif step_num:
                            step_text = f"Bước {step_num}: {step_content}"
                        else:
                            step_text = step_content
                        steps.append(step_text)
            
            # Fallback
            if not steps:
                for selector in ['ol.steps li', 'div.cachlam li', 'div.perform p']:
                    items = soup.select(selector)
                    if items:
                        for i, item in enumerate(items, 1):
                            text = safe_text(item)
                            if text and len(text) > 10:
                                text = re.sub(r'^(Bước\s*)?\d+[.:]?\s*', '', text)
                                steps.append(f"Bước {i}: {text}")
                        if steps:
                            break
            
            result['step'] = steps
        except:
            pass
        
        # Notes - div.tipsrecipe
        try:
            notes = []
            tips_divs = soup.select('div.tipsrecipe, div.infobox, div.tips')
            for tips in tips_divs:
                text = safe_text(tips)
                if text:
                    # Loại bỏ "Mách nhỏ:" prefix
                    text = re.sub(r'^Mách nhỏ:\s*', '', text)
                    notes.append(text)
            result['note'] = notes
        except:
            pass
        
        # Post date - thử tìm từ nhiều nguồn
        try:
            # Từ meta hoặc span.date
            date_tag = soup.find('span', class_='date') or soup.find('time')
            if date_tag:
                result['post_date'] = safe_text(date_tag)
        except:
            pass
        
        return result
        
    except Exception as e:
        print(f"  [ERROR] {url}: {e}")
        return result

In [9]:
# Test
if len(all_recipe_urls) > 0:
    test_cat, test_url = all_recipe_urls[0]
    print(f"Testing: {test_url}")
    detail = get_recipe_detail_dmx(test_url, test_cat)
    print(f"\nTitle: {detail['title']}")
    print(f"Description: {detail['description'][:100] if detail['description'] else 'N/A'}...")
    print(f"Ingredients ({len(detail['ingredients'])}): {detail['ingredients'][:3]}")
    print(f"Steps ({len(detail['step'])}): {detail['step'][:1] if detail['step'] else 'N/A'}")

Testing: https://www.dienmayxanh.com/vao-bep/cach-lam-goi-du-du-tom-kho-hap-dan-don-gian-14052

Title: Cách làm gỏi đu đủ tôm khô hấp dẫn, đơn giản
Description: Nếu bạn đang tìm mộtmón gỏivừa có thể ăn vặt vừa có thể nhâm nhi vào dịp cuối tuần nhưng lại vô cùng...
Ingredients (7): ['Đu đủ xanh - 1 trái', 'Tôm khô - 100 gr', 'Chanh - 1 trái']
Steps (6): ['Bước 1 - Sơ chế nguyên liệu: Đu đủ mua về bạn cắt đôi, bỏ phần hạt bên trong đu đủ rồi dùng dụng cụ bào sợi, bào bên trong của trái đu đủ cho đến khi hết phần thịt. Tiếp theo, bạn cho đu đủ đã bào vào thau nước lạnh, nhồi rửa đu đủ rồi vớt ra rổ để ráo nước. Rau răm mua về bạn nhặt lấy lá, sau đó rửa sạch lại với nước rồi vớt ra để ráo. Nếu thích ăn rau răm nhỏ thì bạn có thể cắt làm đôi hoặc ba đều được nhé!']

Title: Cách làm gỏi đu đủ tôm khô hấp dẫn, đơn giản
Description: Nếu bạn đang tìm mộtmón gỏivừa có thể ăn vặt vừa có thể nhâm nhi vào dịp cuối tuần nhưng lại vô cùng...
Ingredients (7): ['Đu đủ xanh - 1 trái', 'Tôm khô - 100 gr

In [10]:
# Load URLs từ file
if os.path.exists('dmx_recipe_urls.csv'):
    urls_df = pd.read_csv('dmx_recipe_urls.csv')
    all_recipe_urls = list(zip(urls_df['category'], urls_df['url']))
    print(f"Loaded {len(all_recipe_urls)} URLs")

Loaded 269 URLs


In [11]:
# Crawl chi tiết
all_recipes = []
failed_urls = []
CHECKPOINT = 100

for i, (category, url) in enumerate(tqdm(all_recipe_urls, desc="Crawling")):
    try:
        detail = get_recipe_detail_dmx(url, category)
        
        if detail['title'] and len(detail['ingredients']) > 0:
            all_recipes.append(detail)
        else:
            failed_urls.append((category, url, "Missing data"))
        
        if (i + 1) % CHECKPOINT == 0:
            pd.DataFrame(all_recipes).to_csv('dmx_recipes_checkpoint.csv', index=False)
            print(f"\n  💾 Checkpoint: {len(all_recipes)} recipes")
        
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        
    except Exception as e:
        failed_urls.append((category, url, str(e)))

print(f"\n✅ Success: {len(all_recipes)}")
print(f"❌ Failed: {len(failed_urls)}")

Crawling:  37%|███▋      | 99/269 [02:43<04:29,  1.59s/it]


  💾 Checkpoint: 90 recipes


Crawling:  74%|███████▍  | 199/269 [05:28<01:44,  1.49s/it]


  💾 Checkpoint: 186 recipes


Crawling: 100%|██████████| 269/269 [07:16<00:00,  1.62s/it]


✅ Success: 252
❌ Failed: 17


In [12]:
# Lưu kết quả
dmx_df = pd.DataFrame(all_recipes)
dmx_df.to_csv('dmx_recipes_detail.csv', index=False)
print(f"Saved {len(dmx_df)} recipes")

if failed_urls:
    pd.DataFrame(failed_urls, columns=['category', 'url', 'error']).to_csv('dmx_failed_urls.csv', index=False)

Saved 252 recipes


## 4. Merge tất cả nguồn dữ liệu

In [13]:
# Load tất cả sources
dataframes = []

# VnExpress
if os.path.exists('vnexpress_foods_detail_merged.csv'):
    vne_df = pd.read_csv('vnexpress_foods_detail_merged.csv')
    vne_df['source'] = 'vnexpress.net'
    dataframes.append(vne_df)
    print(f"VnExpress: {len(vne_df)} recipes")

# Cooky
if os.path.exists('cooky_recipes_detail.csv'):
    cooky_df = pd.read_csv('cooky_recipes_detail.csv')
    dataframes.append(cooky_df)
    print(f"Cooky: {len(cooky_df)} recipes")

# DMX
if os.path.exists('dmx_recipes_detail.csv'):
    dmx_df = pd.read_csv('dmx_recipes_detail.csv')
    dataframes.append(dmx_df)
    print(f"DMX: {len(dmx_df)} recipes")

VnExpress: 893 recipes
DMX: 252 recipes


In [14]:
# Chuẩn hóa cột
common_cols = [
    'link', 'type_of_food', 'title', 'description', 'author_name',
    'cook_time', 'num_of_people', 'calories', 'num_of_ingredients',
    'ingredients', 'step', 'note', 'post_date', 'source'
]

for i, df in enumerate(dataframes):
    for col in common_cols:
        if col not in df.columns:
            df[col] = None
    dataframes[i] = df[common_cols]

In [15]:
# Merge
if dataframes:
    merged_df = pd.concat(dataframes, ignore_index=True)
    merged_df = merged_df.drop_duplicates(subset=['title'], keep='first')
    
    print(f"\n📊 Final Merged Dataset:")
    print(f"  Total: {len(merged_df)} recipes")
    print(f"\n  By source:")
    print(merged_df['source'].value_counts())
    
    # Lưu
    merged_df.to_csv('all_recipes_final.csv', index=False)
    print(f"\n✅ Saved to all_recipes_final.csv")


📊 Final Merged Dataset:
  Total: 1138 recipes

  By source:
source
vnexpress.net      886
dienmayxanh.com    252
Name: count, dtype: int64

✅ Saved to all_recipes_final.csv

✅ Saved to all_recipes_final.csv


In [16]:
# Thống kê cuối cùng
print("📈 Thống kê loại món:")
print(merged_df['type_of_food'].value_counts().head(20))

📈 Thống kê loại món:
type_of_food
Món ngon hàng ngày             486
Món ngon cho cuối tuần         123
Món Tết                         81
Món tráng miệng, giải khát      56
Quà - Món ăn vặt                51
Món ngon theo vùng miền         24
Món Chay                        24
Món Chè                         23
Món Nướng                       22
Món Hấp                         22
Món Gỏi                         21
Món ngon ngày lạnh              21
Món Kho                         20
Món Xào                         20
Món Chiên                       20
Món Lẩu                         19
Thực đơn cho ngày nắng nóng     19
Món Bánh                        18
Món Thịt Heo                    16
Món Canh                        15
Name: count, dtype: int64
